# Regime Detection Notebook

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from regime_detection.data.ingestion import get_merged_df
from regime_detection.features.features import feature_pipeline
from regime_detection.models.stationarity import stationarity_pipeline


## Config

In [2]:
seed = 42

In [3]:
df_raw = get_merged_df()

df_features = feature_pipeline(df_raw)

split_ratio = .8
split_index = int(len(df_features) * split_ratio)

df_train , df_test = df_features.iloc[:split_index, :], df_features.iloc[split_index:, :]

df_test.head()

Computing log returns...
Computing week-on-week changes...
Computing lagged values...
Computing rolling returns volatility...
Computing commodity-to-commodity differences...
Computing rolling window z-scores...


,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,days_since_opec,days_since_fomc,RBRTE_log_return,...,RBRTE_log_return_12_returns_rol_vol,RWTC_log_return_4_returns_rol_vol,RWTC_log_return_12_returns_rol_vol,RNGWHHD_log_return_4_returns_rol_vol,RNGWHHD_log_return_12_returns_rol_vol,RBRTE_RWTC_diff,RBRTE_log_return_rol_z_score,RWTC_log_return_rol_z_score,RNGWHHD_log_return_rol_z_score,RBRTE_RWTC_diff_rol_z_score
period,,,,,,,,,,,,,,,,,,,,,
2022-05-13,107.43,105.00,0.32,28.87,120.8442,420820,7.47,163.0,9,-0.026908,...,0.080870,0.026163,0.078205,0.109138,0.068749,2.43,-0.756307,-0.518848,-1.056496,-0.486702
2022-05-20,112.93,112.18,0.18,29.43,119.3677,419801,8.21,170.0,16,0.049929,...,0.081771,0.039479,0.079562,0.113081,0.070262,0.75,0.896917,1.201143,0.915620,-1.326487
2022-05-27,117.66,113.38,0.27,25.72,118.2834,414733,8.79,177.0,23,0.041031,...,0.071729,0.035174,0.068082,0.100454,0.067917,4.28,0.700110,0.038517,0.584460,0.492435
2022-06-03,124.11,116.37,0.30,24.79,118.2153,416758,8.54,184.0,30,0.053369,...,0.070760,0.034328,0.066279,0.079397,0.071807,7.74,0.936986,0.348445,-0.573108,2.156033
2022-06-10,127.40,120.43,0.09,27.75,119.9981,418714,8.95,191.0,37,0.026163,...,0.061693,0.023397,0.053998,0.053057,0.069799,6.97,0.326270,0.501446,0.319913,1.699102


In [19]:
df_train = stationarity_pipeline(df_train)

RBRTE_log_return was transformed using none and it's stationarity class is now stationary
RWTC_log_return was transformed using none and it's stationarity class is now stationary
RNGWHHD_log_return was transformed using none and it's stationarity class is now stationary
WCESTUS1_1_w_change was transformed using none and it's stationarity class is now stationary
T10Y2Y was transformed using differenced and it's stationarity class is now stationary
VIXCLS was transformed using differenced and it's stationarity class is now stationary
DTWEXBGS was transformed using differenced and it's stationarity class is now stationary


In [45]:



df_model = df_train.dropna()

#X = df_model[['WCESTUS1_1_w_change_stationary', 'days_since_opec', 'days_since_fomc']]

X = df_model[[
    'WCESTUS1_1_w_change_stationary',  # inventory surprise
    'RBRTE_RWTC_diff',                  # brent-wti spread
    'VIXCLS',                           # risk sentiment
    'T10Y2Y_stationary',                # yield curve
    'DTWEXBGS_stationary',              # dollar index
]]
X = sm.add_constant(X)


wti_y = df_model['RWTC_log_return_stationary'].values.reshape(-1,1)


brent_y = df_model['RBRTE_log_return_stationary'].values.reshape(-1,1)

henry_y = df_model['RNGWHHD_log_return_stationary'].values.reshape(-1,1)

wti_model = sm.OLS(wti_y,X)
brent_model = sm.OLS(brent_y,X)
henry_model = sm.OLS(henry_y, X)

wti_results = wti_model.fit()
brent_results = brent_model.fit()
henry_results = henry_model.fit()

In [49]:
print(f"Henry Hub:\n{henry_results.tvalues}\n")
print(f"Brent: \n {brent_results.tvalues}\n")
print(f"WTI: \n {wti_results.tvalues}\n")

Henry Hub:
const                             1.129299
WCESTUS1_1_w_change_stationary   -3.017639
RBRTE_RWTC_diff                  -0.087769
VIXCLS                           -1.263062
T10Y2Y_stationary                 1.938137
DTWEXBGS_stationary              -1.503226
dtype: float64

Brent: 
 const                             4.385039
WCESTUS1_1_w_change_stationary   -1.239956
RBRTE_RWTC_diff                  -0.315936
VIXCLS                           -4.853372
T10Y2Y_stationary                 3.206614
DTWEXBGS_stationary              -8.921573
dtype: float64

WTI: 
 const                             2.475419
WCESTUS1_1_w_change_stationary   -1.091510
RBRTE_RWTC_diff                  -1.515771
VIXCLS                           -2.277899
T10Y2Y_stationary                 3.152962
DTWEXBGS_stationary              -5.444495
dtype: float64



# interpret test statistics

In [50]:
print(f"Henry Hub:\n{henry_results.params}\n")
print(f"Brent: \n {brent_results.params}\n")
print(f"WTI: \n {wti_results.params}\n")

Henry Hub:
const                             0.010414
WCESTUS1_1_w_change_stationary   -0.000002
RBRTE_RWTC_diff                  -0.000048
VIXCLS                           -0.000479
T10Y2Y_stationary                 0.077412
DTWEXBGS_stationary              -0.006368
dtype: float64

Brent: 
 const                             1.949007e-02
WCESTUS1_1_w_change_stationary   -4.425703e-07
RBRTE_RWTC_diff                  -8.255816e-05
VIXCLS                           -8.877980e-04
T10Y2Y_stationary                 6.173350e-02
DTWEXBGS_stationary              -1.821659e-02
dtype: float64

WTI: 
 const                             2.253504e-02
WCESTUS1_1_w_change_stationary   -7.979459e-07
RBRTE_RWTC_diff                  -8.112682e-04
VIXCLS                           -8.534436e-04
T10Y2Y_stationary                 1.243262e-01
DTWEXBGS_stationary              -2.276947e-02
dtype: float64



In [51]:
print(f"Henry Hub R²: {henry_results.rsquared:.4f}")
print(f"Brent R²: {brent_results.rsquared:.4f}")
print(f"WTI R²: {wti_results.rsquared:.4f}")

Henry Hub R²: 0.0236
Brent R²: 0.1540
WTI R²: 0.0687


In [57]:
henry_resid = henry_results.resid
brent_resid = brent_results.resid
wti_resid = wti_results.resid

print(f"WTI Residuals mean: {wti_resid.mean():.6f}, std: {wti_resid.std():.4f}")
print(f"Brent Residuals mean: {brent_resid.mean():.6f}, std: {brent_resid.std():.4f}")
print(f"Henry Hub Residuals mean: {henry_resid.mean():.6f}, std: {henry_resid.std():.4f}")

WTI Residuals mean: -0.000000, std: 0.0948
Brent Residuals mean: -0.000000, std: 0.0463
Henry Hub Residuals mean: -0.000000, std: 0.0960


Observation: The residuals are centered at zero, so we note that there is no bias when it comes to our predictions.